# Tutorial Completo de Gemini 3 Pro

Bienvenido al tutorial completo de **Gemini 3**, el modelo más avanzado de Google AI hasta la fecha. Este notebook cubre todas las características nuevas y capacidades del modelo Gemini 3.

## Características principales de Gemini 3:

1. **Dynamic Thinking**: Razonamiento avanzado con niveles configurables
2. **Media Resolution Control**: Control granular sobre la resolución de imágenes y videos
3. **Thought Signatures**: Mantenimiento del contexto de razonamiento
4. **Structured Outputs**: Salidas estructuradas con herramientas integradas
5. **Function Calling**: Llamadas a funciones mejoradas
6. **Image Generation**: Generación y edición de imágenes de alta calidad
7. **Multimodal Capabilities**: Procesamiento avanzado de texto, imágenes y video

---

## Modelos disponibles:

| Modelo | Context Window | Knowledge Cutoff | Precio (Input/Output) |
|--------|----------------|------------------|----------------------|
| `gemini-3-pro-preview` | 1M / 64k | Enero 2025 | $2 / $12 (<200k tokens) |
| `gemini-3-pro-image-preview` | 65k / 32k | Enero 2025 | $2 (Text) / $0.134 (Image) |

---

## 1. Configuración Inicial

Primero, instalamos y configuramos las librerías necesarias.

In [ ]:
# Instalar la última versión del SDK de Google Gen AI
!pip install -U google-genai
!pip install pillow requests python-dotenv

In [ ]:
# Imports necesarios
from google import genai
from google.genai import types
import base64
import os
from PIL import Image
import requests
from io import BytesIO
from pydantic import BaseModel, Field
from typing import List, Optional
import json

In [ ]:
# Configurar API Key
# Obtén tu API key desde: https://aistudio.google.com/app/apikey

from google.colab import userdata
# En Colab, guarda tu API key en Secrets con el nombre GOOGLE_API_KEY
API_KEY = userdata.get('GOOGLE_API_KEY')

# Alternativa: usar una variable de entorno
# API_KEY = os.getenv('GOOGLE_API_KEY')

# O directamente (no recomendado para producción)
# API_KEY = 'tu-api-key-aqui'

# Crear el cliente
client = genai.Client(api_key=API_KEY)
print("Cliente de Gemini 3 configurado correctamente!")

## 2. Dynamic Thinking - Niveles de Razonamiento

Gemini 3 introduce el parámetro `thinking_level` que controla la profundidad del razonamiento interno del modelo.

### Niveles disponibles:
- **`low`**: Minimiza latencia y costo. Ideal para tareas simples, chat o alta concurrencia
- **`medium`**: Balance entre razonamiento y velocidad (próximamente)
- **`high`** (por defecto): Maximiza el razonamiento. Ideal para tareas complejas

### Ejemplo 1: Análisis de Algoritmo Complejo

In [ ]:
# Ejemplo con thinking_level HIGH para análisis complejo
codigo_complejo = """
def fibonacci_optimizado(n, memo={}):
    if n in memo:
        return memo[n]
    if n <= 1:
        return n
    memo[n] = fibonacci_optimizado(n-1, memo) + fibonacci_optimizado(n-2, memo)
    return memo[n]
"""

response = client.models.generate_content(
    model="gemini-3-pro-preview",
    contents=f"""
    Analiza el siguiente código Python y:
    1. Explica su complejidad temporal y espacial
    2. Identifica posibles problemas o bugs
    3. Sugiere mejoras de optimización
    4. Compara con otras implementaciones
    
    Código:
    {codigo_complejo}
    """,
    config=types.GenerateContentConfig(
        thinking_level="high"  # Razonamiento profundo
    )
)

print("=== ANÁLISIS CON THINKING LEVEL: HIGH ===")
print(response.text)

### Ejemplo 2: Chat Simple con Thinking Low

In [ ]:
# Ejemplo con thinking_level LOW para respuesta rápida
response_low = client.models.generate_content(
    model="gemini-3-pro-preview",
    contents="¿Cuál es la capital de Francia?",
    config=types.GenerateContentConfig(
        thinking_level="low"  # Respuesta rápida sin razonamiento profundo
    )
)

print("=== RESPUESTA CON THINKING LEVEL: LOW ===")
print(response_low.text)

### Ejemplo 3: Comparación de Thinking Levels

In [ ]:
import time

problema_matematico = """
Un tren sale de la estación A hacia la estación B a 80 km/h. 
30 minutos después, otro tren sale de B hacia A a 100 km/h.
Si la distancia entre A y B es de 300 km, ¿en qué momento se encuentran?
Muestra el proceso de razonamiento paso a paso.
"""

# Test con LOW
start_low = time.time()
response_low = client.models.generate_content(
    model="gemini-3-pro-preview",
    contents=problema_matematico,
    config=types.GenerateContentConfig(thinking_level="low")
)
time_low = time.time() - start_low

# Test con HIGH
start_high = time.time()
response_high = client.models.generate_content(
    model="gemini-3-pro-preview",
    contents=problema_matematico,
    config=types.GenerateContentConfig(thinking_level="high")
)
time_high = time.time() - start_high

print(f"\n{'='*60}")
print("RESPUESTA CON THINKING LOW")
print(f"Tiempo: {time_low:.2f} segundos")
print(f"{'='*60}")
print(response_low.text)

print(f"\n{'='*60}")
print("RESPUESTA CON THINKING HIGH")
print(f"Tiempo: {time_high:.2f} segundos")
print(f"{'='*60}")
print(response_high.text)

## 3. Media Resolution - Control de Resolución

Gemini 3 permite controlar la resolución de procesamiento de imágenes y videos mediante el parámetro `media_resolution`.

### Niveles de resolución:
- **`media_resolution_low`**: 280 tokens por imagen / 70 tokens por frame
- **`media_resolution_medium`**: 560 tokens por imagen / 70 tokens por frame  
- **`media_resolution_high`**: 1120 tokens por imagen / 280 tokens por frame

### Recomendaciones:
- **Imágenes**: `media_resolution_high` (calidad máxima)
- **PDFs**: `media_resolution_medium` (óptimo para documentos)
- **Videos**: `media_resolution_low` o `medium` (suficiente para la mayoría)
- **Videos con texto denso**: `media_resolution_high` (para OCR)

### Ejemplo: Análisis de Imagen con Diferentes Resoluciones

In [ ]:
# Descargar una imagen de ejemplo con texto pequeño
# Usaremos una infografía con texto detallado
image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3f/JPEG_example_flower.jpg/640px-JPEG_example_flower.jpg"

response_img = requests.get(image_url)
image = Image.open(BytesIO(response_img.content))
image.save('test_image.jpg')

# Mostrar la imagen
display(image)
print(f"Tamaño de la imagen: {image.size}")

In [ ]:
# IMPORTANTE: media_resolution solo está disponible en v1alpha
client_alpha = genai.Client(
    api_key=API_KEY,
    http_options={'api_version': 'v1alpha'}
)

# Convertir imagen a base64
with open('test_image.jpg', 'rb') as f:
    image_data = base64.b64encode(f.read()).decode('utf-8')

prompt_vision = "Describe esta imagen en detalle. Menciona colores, objetos, composición y cualquier texto visible."

# Test con diferentes resoluciones
resolutions = ['media_resolution_low', 'media_resolution_medium', 'media_resolution_high']

for resolution in resolutions:
    response = client_alpha.models.generate_content(
        model="gemini-3-pro-preview",
        contents=[
            types.Content(
                parts=[
                    types.Part(text=prompt_vision),
                    types.Part(
                        inline_data=types.Blob(
                            mime_type="image/jpeg",
                            data=base64.b64decode(image_data)
                        ),
                        media_resolution={"level": resolution}
                    )
                ]
            )
        ]
    )
    
    print(f"\n{'='*60}")
    print(f"ANÁLISIS CON {resolution.upper()}")
    print(f"{'='*60}")
    print(response.text)
    print(f"\nTokens usados (aprox): {len(response.text.split())}")

## 4. Function Calling - Llamadas a Funciones

Gemini 3 mejora las capacidades de Function Calling, permitiendo que el modelo llame a funciones personalizadas.

### Ejemplo: Sistema de Gestión de Tareas

In [ ]:
# Definir funciones simuladas
tasks_db = []

def crear_tarea(titulo: str, descripcion: str, prioridad: str) -> str:
    """Crea una nueva tarea en el sistema"""
    task_id = len(tasks_db) + 1
    task = {
        "id": task_id,
        "titulo": titulo,
        "descripcion": descripcion,
        "prioridad": prioridad,
        "completada": False
    }
    tasks_db.append(task)
    return json.dumps({"success": True, "task_id": task_id, "message": f"Tarea '{titulo}' creada exitosamente"})

def listar_tareas() -> str:
    """Lista todas las tareas del sistema"""
    return json.dumps({"tasks": tasks_db, "total": len(tasks_db)})

def completar_tarea(task_id: int) -> str:
    """Marca una tarea como completada"""
    for task in tasks_db:
        if task["id"] == task_id:
            task["completada"] = True
            return json.dumps({"success": True, "message": f"Tarea {task_id} completada"})
    return json.dumps({"success": False, "message": f"Tarea {task_id} no encontrada"})

def obtener_estadisticas() -> str:
    """Obtiene estadísticas de las tareas"""
    total = len(tasks_db)
    completadas = sum(1 for t in tasks_db if t["completada"])
    pendientes = total - completadas
    return json.dumps({
        "total": total,
        "completadas": completadas,
        "pendientes": pendientes
    })

# Definir las declaraciones de funciones para el modelo
function_declarations = [
    types.FunctionDeclaration(
        name="crear_tarea",
        description="Crea una nueva tarea en el sistema de gestión",
        parameters={
            "type": "object",
            "properties": {
                "titulo": {
                    "type": "string",
                    "description": "Título de la tarea"
                },
                "descripcion": {
                    "type": "string",
                    "description": "Descripción detallada de la tarea"
                },
                "prioridad": {
                    "type": "string",
                    "enum": ["baja", "media", "alta"],
                    "description": "Nivel de prioridad de la tarea"
                }
            },
            "required": ["titulo", "descripcion", "prioridad"]
        }
    ),
    types.FunctionDeclaration(
        name="listar_tareas",
        description="Lista todas las tareas existentes en el sistema",
        parameters={"type": "object", "properties": {}}
    ),
    types.FunctionDeclaration(
        name="completar_tarea",
        description="Marca una tarea como completada",
        parameters={
            "type": "object",
            "properties": {
                "task_id": {
                    "type": "integer",
                    "description": "ID de la tarea a completar"
                }
            },
            "required": ["task_id"]
        }
    ),
    types.FunctionDeclaration(
        name="obtener_estadisticas",
        description="Obtiene estadísticas sobre las tareas del sistema",
        parameters={"type": "object", "properties": {}}
    )
]

# Mapeo de funciones
functions_map = {
    "crear_tarea": crear_tarea,
    "listar_tareas": listar_tareas,
    "completar_tarea": completar_tarea,
    "obtener_estadisticas": obtener_estadisticas
}

print("Sistema de funciones configurado correctamente!")

In [ ]:
# Función helper para manejar function calling
def chat_with_functions(user_message, history=None):
    if history is None:
        history = []
    
    # Agregar mensaje del usuario
    history.append(types.Content(
        role="user",
        parts=[types.Part(text=user_message)]
    ))
    
    # Generar respuesta
    response = client.models.generate_content(
        model="gemini-3-pro-preview",
        contents=history,
        config=types.GenerateContentConfig(
            tools=[types.Tool(function_declarations=function_declarations)]
        )
    )
    
    # Procesar función calls
    function_calls = []
    for part in response.candidates[0].content.parts:
        if hasattr(part, 'function_call') and part.function_call:
            function_calls.append(part)
    
    if function_calls:
        # Agregar la respuesta del modelo al historial
        history.append(response.candidates[0].content)
        
        # Ejecutar funciones
        function_responses = []
        for fc_part in function_calls:
            fc = fc_part.function_call
            print(f"\n🔧 Ejecutando función: {fc.name}")
            print(f"   Argumentos: {dict(fc.args)}")
            
            # Llamar a la función
            func = functions_map[fc.name]
            result = func(**dict(fc.args))
            print(f"   Resultado: {result}")
            
            function_responses.append(
                types.Part(
                    function_response=types.FunctionResponse(
                        name=fc.name,
                        response={"result": result}
                    )
                )
            )
        
        # Agregar respuestas de funciones
        history.append(types.Content(
            role="user",
            parts=function_responses
        ))
        
        # Obtener respuesta final
        final_response = client.models.generate_content(
            model="gemini-3-pro-preview",
            contents=history,
            config=types.GenerateContentConfig(
                tools=[types.Tool(function_declarations=function_declarations)]
            )
        )
        
        history.append(final_response.candidates[0].content)
        return final_response.text, history
    else:
        history.append(response.candidates[0].content)
        return response.text, history

# Test del sistema
print("=" * 80)
print("SISTEMA DE GESTIÓN DE TAREAS CON FUNCTION CALLING")
print("=" * 80)

chat_history = []

# Conversación 1: Crear tareas
print("\n📝 Usuario: Crea tres tareas: 1) Estudiar Gemini 3 (alta prioridad), 2) Hacer ejercicio (media), 3) Leer un libro (baja)")
response1, chat_history = chat_with_functions(
    "Crea tres tareas: 1) Estudiar Gemini 3 con descripción 'Revisar todas las características nuevas' (alta prioridad), 2) Hacer ejercicio con descripción 'Correr 30 minutos' (media prioridad), 3) Leer un libro con descripción 'Leer 50 páginas' (baja prioridad)",
    chat_history
)
print(f"\n🤖 Gemini: {response1}")

# Conversación 2: Listar tareas
print("\n" + "=" * 80)
print("\n📝 Usuario: ¿Cuántas tareas tengo y cuáles son?")
response2, chat_history = chat_with_functions(
    "¿Cuántas tareas tengo y cuáles son?",
    chat_history
)
print(f"\n🤖 Gemini: {response2}")

# Conversación 3: Completar tarea
print("\n" + "=" * 80)
print("\n📝 Usuario: Completa la tarea de estudiar Gemini 3")
response3, chat_history = chat_with_functions(
    "Completa la tarea de estudiar Gemini 3",
    chat_history
)
print(f"\n🤖 Gemini: {response3}")

# Conversación 4: Estadísticas
print("\n" + "=" * 80)
print("\n📝 Usuario: Dame estadísticas de mis tareas")
response4, chat_history = chat_with_functions(
    "Dame estadísticas de mis tareas y un resumen de mi progreso",
    chat_history
)
print(f"\n🤖 Gemini: {response4}")

## 5. Structured Outputs - Salidas Estructuradas

Gemini 3 permite combinar structured outputs con herramientas como Google Search, permitiendo extraer información estructurada de fuentes en tiempo real.

### Ejemplo: Extracción de Información de Películas

In [ ]:
# Definir schemas con Pydantic
class Actor(BaseModel):
    nombre: str = Field(description="Nombre completo del actor")
    personaje: str = Field(description="Personaje que interpreta")

class Pelicula(BaseModel):
    titulo: str = Field(description="Título de la película")
    director: str = Field(description="Director de la película")
    año: int = Field(description="Año de estreno")
    generos: List[str] = Field(description="Géneros de la película")
    actores_principales: List[Actor] = Field(description="Lista de actores principales")
    sinopsis: str = Field(description="Sinopsis breve de la película")
    calificacion: Optional[float] = Field(description="Calificación promedio (0-10)")
    presupuesto: Optional[str] = Field(description="Presupuesto de la película")
    recaudacion: Optional[str] = Field(description="Recaudación en taquilla")

# Usar Google Search con structured output
response = client.models.generate_content(
    model="gemini-3-pro-preview",
    contents="Busca información detallada sobre la película 'Oppenheimer' de 2023",
    config=types.GenerateContentConfig(
        tools=[types.Tool(google_search=types.GoogleSearch())],
        response_mime_type="application/json",
        response_schema=Pelicula.model_json_schema()
    )
)

# Parsear el resultado
pelicula = Pelicula.model_validate_json(response.text)

# Mostrar resultados formateados
print("=" * 80)
print("INFORMACIÓN DE LA PELÍCULA (Extraída con Google Search + Structured Output)")
print("=" * 80)
print(f"\n🎬 Título: {pelicula.titulo}")
print(f"🎭 Director: {pelicula.director}")
print(f"📅 Año: {pelicula.año}")
print(f"🎪 Géneros: {', '.join(pelicula.generos)}")
print(f"⭐ Calificación: {pelicula.calificacion}/10")
print(f"💰 Presupuesto: {pelicula.presupuesto}")
print(f"💵 Recaudación: {pelicula.recaudacion}")
print(f"\n📝 Sinopsis:\n{pelicula.sinopsis}")
print(f"\n👥 Actores Principales:")
for actor in pelicula.actores_principales:
    print(f"   - {actor.nombre} como {actor.personaje}")

# Mostrar el JSON estructurado
print(f"\n\n📦 JSON Estructurado:")
print(json.dumps(json.loads(response.text), indent=2, ensure_ascii=False))

### Ejemplo 2: Análisis de Noticias Tecnológicas

In [ ]:
class NoticiasTech(BaseModel):
    tema: str = Field(description="Tema principal de búsqueda")
    fecha_busqueda: str = Field(description="Fecha de la búsqueda")
    noticias: List[dict] = Field(description="Lista de noticias encontradas")
    resumen_general: str = Field(description="Resumen general de las tendencias")

response = client.models.generate_content(
    model="gemini-3-pro-preview",
    contents="Busca las últimas noticias sobre inteligencia artificial generativa en 2025. Dame al menos 3 noticias con título, fuente, fecha y resumen breve de cada una.",
    config=types.GenerateContentConfig(
        tools=[types.Tool(google_search=types.GoogleSearch())],
        response_mime_type="application/json",
        response_schema=NoticiasTech.model_json_schema()
    )
)

noticias = NoticiasTech.model_validate_json(response.text)

print("=" * 80)
print(f"📰 NOTICIAS SOBRE: {noticias.tema}")
print(f"📅 Fecha de búsqueda: {noticias.fecha_busqueda}")
print("=" * 80)

for i, noticia in enumerate(noticias.noticias, 1):
    print(f"\n{i}. {noticia.get('titulo', 'Sin título')}")
    print(f"   📌 Fuente: {noticia.get('fuente', 'Desconocida')}")
    print(f"   📅 Fecha: {noticia.get('fecha', 'N/A')}")
    print(f"   📝 {noticia.get('resumen', 'Sin resumen')}")

print(f"\n{'='*80}")
print(f"📊 RESUMEN GENERAL:")
print(f"{'='*80}")
print(noticias.resumen_general)

## 6. Code Execution - Ejecución de Código

Gemini 3 puede ejecutar código Python internamente para resolver problemas complejos.

### Ejemplo: Análisis de Datos y Visualización

In [ ]:
response = client.models.generate_content(
    model="gemini-3-pro-preview",
    contents="""
    Tengo los siguientes datos de ventas mensuales de una tienda:
    Enero: 15000, Febrero: 18000, Marzo: 22000, Abril: 19000, Mayo: 25000, 
    Junio: 28000, Julio: 31000, Agosto: 29000, Septiembre: 33000, Octubre: 35000
    
    Realiza un análisis completo que incluya:
    1. Calcular el promedio, mediana, y desviación estándar
    2. Calcular el crecimiento porcentual mes a mes
    3. Identificar el mes con mayor y menor venta
    4. Hacer una proyección lineal para los próximos 2 meses
    5. Calcular la tendencia general (creciente/decreciente)
    
    Usa Python para hacer todos los cálculos y muestra los resultados de forma clara.
    """,
    config=types.GenerateContentConfig(
        tools=[types.Tool(code_execution=types.CodeExecution())],
        thinking_level="high"
    )
)

print("=" * 80)
print("ANÁLISIS DE DATOS DE VENTAS CON CODE EXECUTION")
print("=" * 80)
print(response.text)

# Mostrar el código ejecutado si está disponible
for part in response.candidates[0].content.parts:
    if hasattr(part, 'executable_code') and part.executable_code:
        print(f"\n{'='*80}")
        print("CÓDIGO EJECUTADO:")
        print(f"{'='*80}")
        print(part.executable_code.code)
    if hasattr(part, 'code_execution_result') and part.code_execution_result:
        print(f"\n{'='*80}")
        print("RESULTADO DE LA EJECUCIÓN:")
        print(f"{'='*80}")
        print(part.code_execution_result.output)

## 7. Image Generation - Generación de Imágenes

Gemini 3 Pro Image permite generar imágenes de alta calidad con resoluciones hasta 4K.

### Características:
- Resoluciones: 2K y 4K
- Aspect ratios: 16:9, 9:16, 1:1, 4:3, 3:4
- Grounding con Google Search
- Edición conversacional

### Ejemplo: Generación de Infografía

In [ ]:
# Generar una infografía con información en tiempo real
response = client.models.generate_content(
    model="gemini-3-pro-image-preview",
    contents="Genera una infografía moderna y profesional sobre 'Los 5 Avances más Importantes en IA en 2025'. Usa colores vibrantes, iconos tecnológicos y un diseño limpio. Incluye título, subtítulos y texto legible.",
    config=types.GenerateContentConfig(
        tools=[types.Tool(google_search=types.GoogleSearch())],
        image_config=types.ImageConfig(
            aspect_ratio="16:9",
            image_size="2K"
        )
    )
)

# Extraer y mostrar la imagen generada
image_parts = [part for part in response.parts if hasattr(part, 'inline_data') and part.inline_data]

if image_parts:
    print("✅ Imagen generada exitosamente!")
    
    # Mostrar el texto de descripción si existe
    text_parts = [part for part in response.parts if hasattr(part, 'text') and part.text]
    if text_parts:
        print(f"\n📝 Descripción:\n{text_parts[0].text}")
    
    # Convertir y mostrar la imagen
    image_data = image_parts[0].inline_data.data
    image = Image.open(BytesIO(image_data))
    
    # Guardar imagen
    image.save('infografia_ia_2025.png')
    print("\n💾 Imagen guardada como: infografia_ia_2025.png")
    print(f"📐 Tamaño: {image.size[0]}x{image.size[1]} pixels")
    
    # Mostrar imagen
    display(image)
else:
    print("❌ No se pudo generar la imagen")

### Ejemplo 2: Edición Conversacional de Imágenes

In [ ]:
# Nota: La edición conversacional requiere mantener el historial con thought signatures

# Primera generación
print("🎨 Generando imagen inicial...")
response1 = client.models.generate_content(
    model="gemini-3-pro-image-preview",
    contents="Genera una imagen de una ciudad futurista al atardecer, con rascacielos iluminados y vehículos voladores. Estilo cyberpunk.",
    config=types.GenerateContentConfig(
        image_config=types.ImageConfig(
            aspect_ratio="16:9",
            image_size="2K"
        )
    )
)

# Mostrar primera imagen
image_parts1 = [part for part in response1.parts if hasattr(part, 'inline_data') and part.inline_data]
if image_parts1:
    print("\n✅ Imagen inicial generada!")
    image1 = Image.open(BytesIO(image_parts1[0].inline_data.data))
    image1.save('ciudad_futurista_1.png')
    print("Primera versión:")
    display(image1)
    
    # Segunda generación (edición) - Requiere pasar el historial completo con thought signatures
    print("\n🎨 Modificando imagen...")
    
    history = [
        types.Content(
            role="user",
            parts=[types.Part(text="Genera una imagen de una ciudad futurista al atardecer, con rascacielos iluminados y vehículos voladores. Estilo cyberpunk.")]
        ),
        response1.candidates[0].content  # Incluye thought signatures automáticamente
    ]
    
    history.append(
        types.Content(
            role="user",
            parts=[types.Part(text="Ahora cambia el atardecer por un amanecer con tonos rosados y dorados. Agrega más neón en los edificios.")]
        )
    )
    
    response2 = client.models.generate_content(
        model="gemini-3-pro-image-preview",
        contents=history,
        config=types.GenerateContentConfig(
            image_config=types.ImageConfig(
                aspect_ratio="16:9",
                image_size="2K"
            )
        )
    )
    
    # Mostrar segunda imagen
    image_parts2 = [part for part in response2.parts if hasattr(part, 'inline_data') and part.inline_data]
    if image_parts2:
        print("\n✅ Imagen editada!")
        image2 = Image.open(BytesIO(image_parts2[0].inline_data.data))
        image2.save('ciudad_futurista_2.png')
        print("\nSegunda versión (editada):")
        display(image2)
else:
    print("❌ No se pudo generar la imagen inicial")

## 8. Análisis Multimodal Avanzado

Gemini 3 puede procesar múltiples tipos de contenido simultáneamente.

### Ejemplo: Análisis de Documentos Técnicos

In [ ]:
# Descargar un PDF de ejemplo (paper técnico)
!wget -q https://arxiv.org/pdf/1706.03762.pdf -O attention_paper.pdf

# Subir el PDF a Gemini
pdf_file = client.files.upload(file='attention_paper.pdf')
print(f"✅ PDF subido: {pdf_file.name}")
print(f"   Estado: {pdf_file.state}")
print(f"   Tamaño: {pdf_file.size_bytes / 1024:.2f} KB")

# Esperar a que el archivo esté listo
import time
while pdf_file.state == 'PROCESSING':
    time.sleep(2)
    pdf_file = client.files.get(name=pdf_file.name)
    print(f"   Estado: {pdf_file.state}")

if pdf_file.state == 'ACTIVE':
    print("\n✅ Archivo listo para análisis")
    
    # Crear un cliente v1alpha para usar media_resolution
    client_alpha = genai.Client(
        api_key=API_KEY,
        http_options={'api_version': 'v1alpha'}
    )
    
    # Analizar el documento con alta resolución
    response = client_alpha.models.generate_content(
        model="gemini-3-pro-preview",
        contents=[
            types.Content(
                parts=[
                    types.Part(text="""Analiza este paper técnico y proporciona:
                    1. Título y autores
                    2. Resumen del problema que resuelve
                    3. Metodología principal (explicada de forma simple)
                    4. Contribuciones clave
                    5. Resultados principales
                    6. Impacto en el campo de IA
                    7. Conceptos técnicos clave que introduce
                    
                    Explica de forma que alguien con conocimientos básicos de ML pueda entender."""),
                    types.Part(
                        file_data=types.FileData(
                            file_uri=pdf_file.uri,
                            mime_type="application/pdf"
                        ),
                        media_resolution={"level": "media_resolution_medium"}
                    )
                ]
            )
        ],
        config=types.GenerateContentConfig(
            thinking_level="high"
        )
    )
    
    print("\n" + "="*80)
    print("ANÁLISIS DEL PAPER TÉCNICO")
    print("="*80)
    print(response.text)
    
    # Limpiar: eliminar el archivo
    client.files.delete(name=pdf_file.name)
    print("\n🗑️ Archivo temporal eliminado")
else:
    print(f"❌ Error procesando el archivo. Estado: {pdf_file.state}")

## 9. Streaming - Respuestas en Tiempo Real

Gemini 3 soporta streaming para respuestas en tiempo real.

### Ejemplo: Generación de Código con Streaming

In [ ]:
from IPython.display import clear_output, Markdown, display
import sys

print("🚀 Generando código en tiempo real...\n")
print("="*80)

accumulated_text = ""

# Usar generate_content_stream para streaming
stream = client.models.generate_content_stream(
    model="gemini-3-pro-preview",
    contents="""Crea una aplicación web completa en Python usando Flask que:
    1. Tenga una página principal con un formulario
    2. Permita subir una imagen
    3. Procese la imagen (redimensionar a 800x600)
    4. Muestre la imagen procesada
    5. Incluya CSS para un diseño moderno
    
    Proporciona todo el código necesario con comentarios explicativos.""",
    config=types.GenerateContentConfig(
        thinking_level="high"
    )
)

for chunk in stream:
    if chunk.text:
        accumulated_text += chunk.text
        # Actualizar la salida (en Colab se ve mejor con clear_output)
        clear_output(wait=True)
        print("="*80)
        print("📝 CÓDIGO GENERADO (Streaming en tiempo real)")
        print("="*80)
        print(accumulated_text)
        sys.stdout.flush()

print("\n\n✅ Generación completada!")

## 10. Mejores Prácticas y Recomendaciones

### 1. Thinking Level
- **Alta complejidad**: Usa `thinking_level="high"` para matemáticas, análisis profundo, debugging
- **Tareas simples**: Usa `thinking_level="low"` para chat, traducción, resúmenes básicos

### 2. Temperature
- **Mantén temperature=1.0** (valor por defecto)
- Cambiar la temperatura puede causar comportamientos inesperados o bucles

### 3. Media Resolution
- **Imágenes**: Usa `media_resolution_high` para máxima calidad
- **PDFs**: Usa `media_resolution_medium` (suficiente para documentos)
- **Videos**: Usa `media_resolution_low` o `medium` para la mayoría de casos

### 4. Prompting
- **Sé conciso y directo**: Gemini 3 responde mejor a instrucciones claras
- **Contexto al final**: Cuando uses documentos largos, coloca tus preguntas al final
- **Usa "Basándote en..."**: Para anclar el razonamiento al contexto proporcionado

### 5. Thought Signatures
- Los SDKs oficiales manejan thought signatures automáticamente
- Son críticos para function calling y edición de imágenes
- Deben incluirse en el historial para mantener el contexto

### Ejemplo de Prompt Optimizado

In [ ]:
# ❌ MALO: Prompt verboso y confuso
bad_prompt = """
Hola, espero que estés teniendo un buen día. Me gustaría que me ayudaras con algo,
si no es mucha molestia. Verás, tengo un código que escribí hace tiempo y no estoy
muy seguro de si está bien o mal, así que si pudieras revisarlo y darme tu opinión
experta sobre qué podría estar mal o cómo podría mejorarlo, te lo agradecería mucho.
El código es el siguiente: [código]
"""

# ✅ BUENO: Prompt claro y directo
good_prompt = """
Revisa este código Python y:
1. Identifica errores
2. Sugiere mejoras
3. Optimiza el rendimiento

[código]
"""

# Demostración
codigo_ejemplo = """
def buscar_elemento(lista, elemento):
    for i in range(len(lista)):
        if lista[i] == elemento:
            return i
    return -1

numeros = [1, 2, 3, 4, 5] * 1000
resultado = buscar_elemento(numeros, 4999)
"""

response = client.models.generate_content(
    model="gemini-3-pro-preview",
    contents=f"""
    Revisa este código Python y:
    1. Identifica errores o ineficiencias
    2. Sugiere mejoras específicas
    3. Proporciona código optimizado
    
    {codigo_ejemplo}
    """,
    config=types.GenerateContentConfig(
        thinking_level="high"
    )
)

print("="*80)
print("ANÁLISIS DE CÓDIGO CON PROMPT OPTIMIZADO")
print("="*80)
print(response.text)

## 11. Conclusión y Recursos

### Resumen de Características de Gemini 3:

✅ **Dynamic Thinking**: Control de profundidad de razonamiento
✅ **Media Resolution**: Control granular de procesamiento multimedia
✅ **Function Calling**: Integración con herramientas personalizadas
✅ **Structured Outputs**: Salidas JSON con schemas definidos
✅ **Image Generation**: Imágenes hasta 4K con edición conversacional
✅ **Google Search**: Grounding con información en tiempo real
✅ **Code Execution**: Ejecución de código Python interno
✅ **Multimodal**: Texto, imágenes, PDFs, videos
✅ **Context Window**: 1M tokens de entrada, 64k de salida

### Recursos Adicionales:

- 📚 [Documentación Oficial](https://ai.google.dev/gemini-api/docs/gemini-3)
- 🔧 [Gemini API Reference](https://ai.google.dev/api)
- 📓 [Gemini Cookbook](https://github.com/google-gemini/cookbook)
- 🎮 [Google AI Studio](https://aistudio.google.com/)
- 💬 [Comunidad de Desarrolladores](https://discuss.ai.google.dev/)

### Próximos Pasos:

1. Experimenta con diferentes `thinking_level` para tus casos de uso
2. Prueba la generación de imágenes con distintas configuraciones
3. Implementa function calling para integrar tus APIs
4. Usa structured outputs para extraer datos de forma confiable
5. Combina múltiples herramientas (Search + Code Execution + Function Calling)

---

**¡Gracias por completar este tutorial!** 🎉

Si tienes preguntas o encuentras problemas, revisa la [documentación oficial](https://ai.google.dev/gemini-api/docs/gemini-3) o la [página de issues](https://github.com/google-gemini/python-genai/issues).

---

*Tutorial creado para Google Colab - Gemini 3 Pro Preview*
*Última actualización: Noviembre 2025*

## 12. Limpieza y Verificación Final

In [ ]:
# Verificar archivos creados
print("📁 Archivos generados en esta sesión:\n")
!ls -lh *.png *.jpg *.pdf 2>/dev/null || echo "No hay archivos generados"

# Listar archivos en Gemini Files API
print("\n☁️ Archivos en Gemini Files API:\n")
try:
    files = client.files.list()
    for file in files:
        print(f"   - {file.name} ({file.size_bytes / 1024:.2f} KB)")
except Exception as e:
    print(f"   No hay archivos o error: {e}")

print("\n✅ Tutorial completado exitosamente!")
print("\n💡 Tips finales:")
print("   - Experimenta con diferentes prompts y parámetros")
print("   - Combina múltiples características para casos de uso complejos")
print("   - Revisa los costos en: https://ai.google.dev/pricing")
print("   - Usa Context Caching para reducir costos con prompts repetitivos")